# Train the glaucoma models on ACRIMA + G1020 (Google Colab)

Not yet run (written without access to Colab). If a cell fails, read its error: most are a missing file or folder
and the message says which. Use **Runtime > Change runtime type > GPU** first.

What is trained on what:
- **ACRIMA**: healthy-only autoencoder (anomaly score), supervised baseline, and the classification test split.
- **G1020** (has disc and cup outlines): trains the segmenter. That segmenter is then run on every ACRIMA image to get its
  cup-to-disc ratio (ACRIMA has no outlines of its own). So the ACRIMA CDR is *predicted*, not measured by an expert.

At the end (step 7) you can open the website inside Colab, using the models you trained.

Screening aid only. Not a medical diagnosis.

In [ ]:
import torch, subprocess
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU (training will be slow)")

## 1. Get the project
Upload `glaucoma-anomaly.zip` (the file you downloaded) when asked.

In [ ]:
from google.colab import files
up = files.upload()                      # choose glaucoma-anomaly.zip
import zipfile, os
zname = next(iter(up))
zipfile.ZipFile(zname).extractall("/content")
%cd /content/glaucoma-anomaly
!pip -q install numpy pandas scikit-learn opencv-python-headless pillow pyyaml matplotlib flask

## 2. Data
**ACRIMA** is open and is attempted automatically. **G1020** you download yourself (links in `docs/DATA.md`).

In [ ]:
!python scripts/download_data.py

If ACRIMA did not download: get it from the figshare link printed above (or the Kaggle mirror), unzip it, and put
the images in `/content/glaucoma-anomaly/data/raw/acrima/`.

**G1020**: download it (https://www.dfki.uni-kl.de/g1020 or the Kaggle copy in `docs/DATA.md`), put the unzipped folder in your
Google Drive, and set `G1020_DRIVE_FOLDER` below to that folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
G1020_DRIVE_FOLDER = "/content/drive/MyDrive/G1020"            # <-- change to where you put it
ACRIMA_DRIVE_FOLDER = None                                   # e.g. "/content/drive/MyDrive/ACRIMA" if you stored it there
import shutil, pathlib
for src, dst in [(G1020_DRIVE_FOLDER, "data/raw/g1020"), (ACRIMA_DRIVE_FOLDER, "data/raw/acrima")]:
    if src:
        assert pathlib.Path(src).exists(), f"Folder not found: {src}"
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("copied", src, "->", dst)

## 3. Check the data before training
Counts must be ACRIMA 705 / 396 / 309, and G1020 must show about 1020 images and about 1020 json files with `disc` and `cup` labels.

In [ ]:
!python scripts/inspect_data.py

Read the output. If G1020 shows 0 json files or no `disc`/`cup` labels, stop and send the output of this cell: the annotation format
differs from what `src/glaucoma/data/seg_prepare.py` expects. Do not train on data the code cannot read.

## 4. Smoke test (about 2 minutes), then the real run

In [ ]:
!python -m unittest discover -s tests 2>&1 | tail -5
!python scripts/run_pipeline.py --quick

In [ ]:
!rm -rf results/quick checkpoints          # remove smoke-test outputs
!python scripts/run_pipeline.py
!python scripts/preview_cdr.py --n 24      # LOOK at results/cdr_preview/*.png: are the outlines on the disc and cup?


## 5. Results
These are the numbers for your paper. They come from the test split only.

In [ ]:
print(open("results/tables/main_results.md").read())
import json; m = json.load(open("results/metrics.json"))
print("Segmentation Dice:", m.get("segmenter"))
print("Mean anomaly score on test (normal vs glaucoma):", m["autoencoder"]["test_mean_error"])
ck = m.get("cdr_check", {})
print("CDR alone, AUC (val / test):", ck.get("val_auc"), "/", ck.get("test_auc"))
print("Mean CDR on validation, normal vs glaucoma:", ck.get("val_mean_cdr_normal"), "vs", ck.get("val_mean_cdr_glaucoma"))
for w in ck.get("warnings", []):
    print("WARNING:", w)

**Check before you trust them:** if the autoencoder AUC is near 0.5 it is not separating glaucoma from healthy
(see `docs/TROUBLESHOOTING.md`). Report that honestly. Also note the confidence intervals, which are wide on small
test sets.

Also read the **CDR check** lines above. ACRIMA has no cup/disc outlines, so the ratio there is a model estimate. If it warns that the ratio is near chance, inverted, or often missing, say so in the report and look at `results/cdr_preview/` (cell 13) before claiming the fusion helps.


## 6. Download the trained models and put them in your project

In [ ]:
!zip -qr trained_models.zip checkpoints results
from google.colab import files
files.download("trained_models.zip")

## 7. Try the website right here in Colab (optional)
This starts the website inside the Colab session and opens it in a new tab. It uses the models you just trained, so run
step 4 (the real run, not only the smoke test) first. Not yet run: written without access to Colab.

- If it says **TEST models**, the checkpoints came from the smoke test. Run the real training cell again.
- Colab makes the link private: it works for you while this notebook is running, not for other people.
- If no tab opens, allow pop-ups for this page, or run the next cell to show the site inside the notebook.

In [ ]:
# Start the website in the background and wait until it answers.
import json, subprocess, sys, time, urllib.request
server = subprocess.Popen([sys.executable, "-m", "backend.app", "--host", "0.0.0.0", "--port", "5000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
health = None
for _ in range(90):
    if server.poll() is not None:
        break                                    # the server stopped: the log printed below says why
    try:
        health = json.load(urllib.request.urlopen("http://127.0.0.1:5000/health", timeout=2))
        break
    except Exception:
        time.sleep(1)
if health is None:
    print(open("server.log").read()[-3000:])
    raise SystemExit("The website did not start. The server log is printed above.")
print("Server is up. models_loaded =", health["models_loaded"], "| test_models =", health.get("test_models"))
if not health["models_loaded"]:
    print("The models are not loaded, so the site opens but cannot analyse images. Reason:", health.get("detail"))
if health.get("test_models"):
    print("WARNING: these are TEST models from the smoke test. Run the real training cell first.")
from google.colab import output
output.serve_kernel_port_as_window(5000, path="/")      # opens the website in a new tab

In [ ]:
# Only if no tab opened: show the website inside this notebook instead.
from google.colab import output
try:
    output.serve_kernel_port_as_iframe(5000, height="900")
except TypeError:
    output.serve_kernel_port_as_iframe(5000)

In [ ]:
# When you are done (this also happens by itself when the Colab session ends):
server.terminate()

## 8. Run the website on your own computer
Extract `trained_models.zip` into the project folder (so `checkpoints/` and `results/` sit next to `backend/`), then:

```
pip install -r requirements.txt
python -m backend.app --open
```

The website opens at http://localhost:5000 (no Node or npm needed). `python scripts/doctor.py` should show the
checkpoints as PASS and the website as served.

Note: the built-in Flask server is meant for demos and class presentations. It is not set up to be put on the open internet.